# FS-DualNet v2 vs DIRE under the same conditions
**Session:** GPU **T4 x2**, **Internet On** (downloads the DIRE code, its ADM diffusion model 2.1 GB, CLIP, two DeepFakeFace zips).
**+ Add Input:** `fs-dualnet-code` (**latest version**), `diffusionforensics`, and **Your Work -> Notebooks -> `fs-dualnet-v2`** (its output holds the v2 checkpoints).
**Run:** Save Version -> **Save & Run All (Commit)**. Estimated 2-3 h. Everything is resumable: if the last cell says NOT FINISHED, commit again with this notebook's own output added as input.

What it measures (3 seeds each, same images for both methods):
1. **Full test sets** with the official precomputed DIRE maps (as in the DIRE paper), and with format-fair maps.
2. **Format-fair maps:** in the official release real DIRE maps are JPEG and fake maps PNG, and the file format alone can decide DIRE's prediction (Ricker et al., CVPR 2024 suppl. Sec. 9). We recompute DIRE with the official recipe (ADM-256, 20+20 DDIM steps) for 400 test images and save real and fake maps the same way (PNG).
3. **Robustness:** JPEG q65, JPEG q30, blur sigma 3 applied to the image before DIRE / before v2.
4. **Cross-dataset:** celebahq-trained models on 400 DeepFakeFace images.
5. **Speed** on the same T4.

In [ ]:
import time
BUDGET_H = 8     # hours this commit may use (Kaggle stops a session at 12 h); everything is resumable
T_START = time.time()
DEADLINE = T_START + BUDGET_H * 3600
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
N_GPU = torch.cuda.device_count()
assert N_GPU >= 1, "No GPU: Settings -> Accelerator -> GPU T4 x2"
print('GPUs:', N_GPU, '(2 = DIRE maps on GPU 1 while the DIRE classifiers train on GPU 0)')

## 1. Find code, data, v2 checkpoints, earlier results of this notebook

In [ ]:
import os, sys, shutil, subprocess, glob, json

INPUT, WORK = '/kaggle/input', '/kaggle/working'

def scan(root, maxdepth=7):
    dirs, codes = {}, []
    base = root.rstrip('/').count('/')
    for dp, dns, fns in os.walk(root):
        for d in dns:
            dirs.setdefault(d.lower(), []).append(os.path.join(dp, d))
        if 'dire_compute.py' in fns and 'train_eval.py' in fns:
            codes.append(dp)
        if dp.count('/') - base >= maxdepth:
            dns[:] = []
    return dirs, codes

def sh(cmd):
    print('$', cmd if len(cmd) < 300 else cmd[:300] + ' ...', flush=True)
    get_ipython().system(cmd)

DIRS, CODES = scan(INPUT)
assert CODES, "new code not found: upload the NEW fs_dualnet.zip as a New Version of 'fs-dualnet-code' and use its latest version"
shutil.copytree(CODES[0], f'{WORK}/fs_dualnet', dirs_exist_ok=True)
os.chdir(f'{WORK}/fs_dualnet')

roots = [os.path.dirname(d) for d in DIRS.get('images', []) if os.path.isdir(os.path.join(os.path.dirname(d), 'dire'))]
assert roots, "data not found: + Add Input -> 'diffusionforensics' (needs both Images/ and dire/)"
RAW_ROOT = roots[0]

v2s = [d for d in DIRS.get('v2_results', []) if os.path.exists(f'{d}/cb_s42/v2.pt')]
assert v2s, "v2 checkpoints not found: + Add Input -> Your Work -> Notebooks -> fs-dualnet-v2"
V2 = v2s[0]

R, MAPS = f'{WORK}/vsdire_results', f'{WORK}/dire_maps'
for name, dst in [('vsdire_results', R), ('dire_maps', MAPS)]:
    prev = DIRS.get(name, [])
    if prev and not os.path.exists(dst):
        shutil.copytree(prev[0], dst, symlinks=True)
        print('RESUMING', name, 'from', prev[0])
os.makedirs(R, exist_ok=True)
SEEDS, EPOCHS = (42, 1, 2), 5
print('code:', CODES[0], '\ndata:', RAW_ROOT, '\nv2  :', V2, '\nout :', R)

In [ ]:
# DIRE's own code (its guided-diffusion copy) + the ADM checkpoint DIRE uses + CLIP for v2
import importlib.util
if importlib.util.find_spec('transformers') is None:
    sh('pip install -q transformers')
sh('pip install -q blobfile')
GD = '/tmp/DIRE/guided-diffusion'
if not os.path.isdir(GD):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/ZhendongWang6/DIRE', '/tmp/DIRE'])
ok = os.path.isdir(GD) and subprocess.run([sys.executable, 'dire_compute.py', 'check_import', '--gd_path', GD]).returncode == 0
if not ok:   # fallback: OpenAI's guided-diffusion (dire_compute.py then runs the same DDIM loop itself)
    sh('pip install -q git+https://github.com/openai/guided-diffusion')
    GD = ''
    assert subprocess.run([sys.executable, 'dire_compute.py', 'check_import']).returncode == 0, 'guided_diffusion not importable'
ADM = '/tmp/adm/256x256_diffusion_uncond.pt'
if not os.path.exists(ADM):
    os.makedirs('/tmp/adm', exist_ok=True)
    r = subprocess.run(['curl', '-fL', '--retry', '3', '-sS', '-o', ADM,
                        'https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt'])
    assert r.returncode == 0 and os.path.getsize(ADM) > 1e9, 'ADM download failed: Settings -> Internet must be On'
print('ADM checkpoint', round(os.path.getsize(ADM) / 2**30, 2), 'GB | guided-diffusion from', GD or 'pip (openai)')
r = subprocess.run([sys.executable, '-c', "from transformers import CLIPVisionModel as M; M.from_pretrained('openai/clip-vit-base-patch16')"],
                   capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]

## 2. Data: official DIRE maps (for the DIRE classifiers), the shared 400 + 400 test images, job list

In [ ]:
sh(f'python prep_data.py --root {RAW_ROOT} --subdir dire --skip stargan --out /tmp/data_dire')
os.makedirs('/tmp/dff', exist_ok=True)
for n in ['wiki', 'text2img']:          # DeepFakeFace: IMDB-WIKI reals and Stable Diffusion text2img fakes
    if not (os.path.isdir(f'/tmp/dff/{n}') and os.listdir(f'/tmp/dff/{n}')):
        r = subprocess.run(['curl', '-fL', '--retry', '3', '-sS', '-o', f'/tmp/dff/{n}.zip',
                            f'https://huggingface.co/datasets/OpenRL/DeepFakeFace/resolve/main/{n}.zip'])
        assert r.returncode == 0, 'DeepFakeFace download failed: Settings -> Internet must be On'
        subprocess.run(['unzip', '-q', '-o', f'/tmp/dff/{n}.zip', '-d', f'/tmp/dff/{n}'], check=True)
        os.remove(f'/tmp/dff/{n}.zip')
JOBS = f'{R}/dire_jobs.txt'
sh(f'python make_dire_subsets.py --raw {RAW_ROOT} --out /tmp/sub --maps {MAPS} --jobs {JOBS} '
   f'--dff_real /tmp/dff/wiki --dff_fake /tmp/dff/text2img')
sh('python cache_resize.py --src /tmp/sub/orig --dst /tmp/sub/rgb --size 256 --procs 4')   # v2 reads 256 caches, like before

## 3. Check the DIRE pipeline on 8 images (maps vs the official maps of the same images) and time it

In [ ]:
jobs = [l for l in open(JOBS) if '\tclean\t' in l and '/dalle2/' in l]
check = [l for l in jobs if '/real/' in l][:4] + [l for l in jobs if '/fake/' in l][:4]
open('/tmp/check_jobs.txt', 'w').writelines(check)
DC = f'python dire_compute.py run --model {ADM} ' + (f'--gd_path {GD} ' if GD else '')
sh(f'CUDA_VISIBLE_DEVICES=0 {DC} --jobs /tmp/check_jobs.txt --lockdir /tmp/locks_check --bs 8')
made = [l.split('\t')[2].strip() for l in check if os.path.exists(l.split('\t')[2].strip())]
assert len(made) == 8, 'DIRE maps were not produced: see the error above'
sh(f'python dire_compute.py validate --ours {MAPS}/clean/dalle2 --official /tmp/sub/official/dalle2 --out {R}/dire_validate.json')
if not os.path.exists(f'{R}/dire_speed.json'):
    sh(f"CUDA_VISIBLE_DEVICES=0 {DC.replace(' run ', ' bench ')} --jobs {JOBS} --bs 16 --out {R}/dire_speed.json")

## 4. DIRE classifiers (GPU 0) while GPU 1 computes the maps
`dire_off` = ResNet-50 on the official maps, as in the paper. `dire_fair` = the same, but every PNG map is re-saved as JPEG q95
when loaded, so real and fake maps have the same format (the official real maps are JPEG q95).

In [ ]:
LOCKS = '/tmp/dire_locks'
shutil.rmtree(LOCKS, ignore_errors=True)
worker = None
if N_GPU >= 2:
    log1 = open(f'{R}/dire_worker_gpu1.log', 'a')
    worker = subprocess.Popen(f'CUDA_VISIBLE_DEVICES=1 {DC} --jobs {JOBS} --lockdir {LOCKS} --deadline {DEADLINE}',
                              shell=True, stdout=log1, stderr=subprocess.STDOUT)
    print('DIRE worker started on GPU 1 (log: dire_worker_gpu1.log)')
D = '/tmp/data_dire'
TESTS = ' '.join(f'{g}={D}/test/{g}' for g in ['dalle2', 'midjourney', 'sdv2'])
for kind, tag, extra in [('dire_off', 'dire_resnet50', ''), ('dire_fair', 'dire_fair', '--png_to_jpeg 95')]:
    for s in SEEDS:
        out = f'{R}/{kind}_s{s}'
        if os.path.exists(f'{out}/results_{tag}.json'):
            print('done:', out); continue
        sh(f'CUDA_VISIBLE_DEVICES=0 python train_eval.py --branch resnet50 --tag {tag} {extra} --train {D}/train/sdv2 '
           f'--val {D}/val/sdv2 --dev dev_if={D}/test/if --test {TESTS} --epochs {EPOCHS} --workers 4 --seed {s} '
           f'--out_dir {out} --deadline {DEADLINE}')

In [ ]:
# GPU 0 helps with the remaining maps, then wait for GPU 1
sh(f'CUDA_VISIBLE_DEVICES=0 {DC} --jobs {JOBS} --lockdir {LOCKS} --deadline {DEADLINE}')
if worker is not None:
    worker.wait()
    print(''.join(open(f'{R}/dire_worker_gpu1.log').readlines()[-3:]))
missing = sum(not os.path.exists(l.split('\t')[2].strip()) for l in open(JOBS))
print('DIRE maps missing:', missing)
sh(f'python dire_compute.py asym --src {MAPS}/clean --dst /tmp/maps_asym')   # diagnostic: official format bias re-created

## 5. Evaluate every model on the same images, then one table

In [ ]:
SETS = ['dalle2', 'midjourney']
CONDS = ['clean', 'jpeg_q65', 'jpeg_q30', 'blur_s3']
E = f'{R}/evals'
def spec(out, pert, root, with_dff=False):
    sets = [f'{g}={root}/{g}' for g in SETS] + ([f'dff={root}/dff'] if with_dff else [])
    return f'--run "{E}/{out}:{pert}:{",".join(sets)}"'
for s in SEEDS:   # v2: RGB images, perturbation applied by the loader (after the 256 resize, before the crop)
    if not os.path.exists(f'{E}/v2_s{s}_blur_s3.npz'):
        runs = ' '.join(spec(f'v2_s{s}_{c}', 'none' if c == 'clean' else c, '/tmp/sub/rgb', c == 'clean') for c in CONDS)
        sh(f'python eval_ckpt.py --ckpt {V2}/cb_s{s}/v2.pt {runs}')
for kind, tag in [('dire_off', 'dire_resnet50'), ('dire_fair', 'dire_fair')]:
    for s in SEEDS:
        ck = f'{R}/{kind}_s{s}/{tag}.pt'
        if not os.path.exists(ck) or os.path.exists(f'{E}/{kind}_s{s}_blur_s3.npz'):
            continue
        runs = [spec(f'{kind}_s{s}_official', 'none', '/tmp/sub/official'), spec(f'{kind}_s{s}_asym', 'none', '/tmp/maps_asym')]
        runs += [spec(f'{kind}_s{s}_{c}', 'none', f'{MAPS}/{c}', c == 'clean') for c in CONDS]
        sh(f'python eval_ckpt.py --ckpt {ck} ' + ' '.join(runs))
sh(f'python compare_v2_dire.py --v2_dir {V2} --res {R}')

In [ ]:
S = '/tmp/vsdire_summary'
shutil.rmtree(S, ignore_errors=True)
for f in glob.glob(f'{R}/**/*', recursive=True):
    if os.path.isfile(f) and f.endswith(('.md', '.json', '.npz', '.log')):
        dst = os.path.join(S, os.path.relpath(f, R)); os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(f, dst)
shutil.make_archive(f'{WORK}/vsdire_summary', 'zip', S)
done_eval = len(glob.glob(f'{R}/evals/*.npz'))
missing = sum(not os.path.exists(l.split('\t')[2].strip()) for l in open(JOBS))
print(f'elapsed {(time.time() - T_START) / 3600:.2f} h | Output tab -> vsdire_summary.zip')
print('ALL FINISHED' if missing == 0 and done_eval >= 3 * 4 + 6 * 6 else
      f'NOT FINISHED (maps missing {missing}, evaluations {done_eval} of 48) -> Save Version again with THIS notebook output as input')